# axon-lang — sincronizar entre Colabs

Cada sessão do Colab enxerga o Drive de uma conta, então os experts, os adapters e os
GGUF acabam espalhados por contas diferentes. Este notebook usa o **git como o lugar
único**: sobe o que existe no Drive desta conta, baixa o que estiver no git.

Duas branches órfãs no repositório de dados — sem histórico comum com a `main`, pra
cada uma pesar só o que guarda:

| branch | vem de | o que é |
|---|---|---|
| `experts` | `MyDrive/axon_experts` | router + base de conhecimento |
| `adapters` | `MyDrive/axon_lora` | adapters LoRA (formato PEFT) |
| `adapters` | `MyDrive/axon_gguf` | os mesmos, convertidos pro Ollama |

**Não precisa de GPU.** É só cópia de arquivo e git.

> **Token:** subir exige um `GH_TOKEN` com permissão de escrita — clonar repositório
> público é livre, mas push não. Guarde nos **Secrets** do Colab (ícone da 🔑, à
> esquerda), com o nome `GH_TOKEN` e o *acesso ao notebook* ligado. Baixar não precisa.

## 1. Preparar

In [ ]:
%cd /content
import os, sys

if os.path.isdir("axon-llm"):
    !git -C axon-llm pull --quiet && echo "repo atualizado"
else:
    !git clone --depth 1 https://github.com/geraldogrise/axon-llm.git axon-llm

sys.path.insert(0, "/content/axon-llm/notebooks")
import axon_sync as sync

# monta o Drive já aqui, pra o status enxergar o que existe nesta conta
from google.colab import drive
drive.mount("/content/drive")

## 2. O que existe aqui e o que já está no git

Roda isto antes de qualquer coisa: mostra, por tipo, o que a conta **desta** sessão tem
no Drive e se a branch correspondente já existe no repositório.

In [ ]:
sync.status()

## 3. Subir — Drive → git

Manda o que esta conta tem pra dentro da branch. Cada tipo substitui apenas a sua
própria pasta, então subir `lora` não apaga o que já estava em `gguf`.

Antes do push ele mede os arquivos: **o GitHub rejeita acima de 100 MB** e avisa acima
de 50 MB. Se algo passar do limite, ele para antes de tentar — um arquivo grande demais
faria o push inteiro falhar depois de já ter transferido o resto.

In [ ]:
sync.subir("experts")

# sync.subir("lora")
# sync.subir("gguf")

## 4. Baixar — git → Drive

Numa conta nova, é isto que reconstrói tudo sem retreinar nada.

Por padrão ele **não sobrescreve** o que já existe no Drive: arquivos com o mesmo nome
são mantidos e ele diz quantos pulou. Use `sobrescrever=True` quando o que está no git
for mais recente.

In [ ]:
sync.baixar("experts")

# sync.baixar("lora")
# sync.baixar("gguf")

# sync.baixar("experts", sobrescrever=True)   # quando o git é a versão boa

## 5. Conferir

Depois de baixar, o `AxonSystem` deve enxergar os experts normalmente. Esta célula
precisa do `pyaxon` compilado — vale rodar só se você for usar o sistema nesta sessão.

In [ ]:
import glob

raiz = "/content/drive/MyDrive/axon_experts"
pastas = sorted(d for d in os.listdir(raiz) if os.path.isdir(os.path.join(raiz, d)))
print(f"{len(pastas)} experts no Drive desta conta:")
for d in pastas:
    n = len(glob.glob(os.path.join(raiz, d, "*")))
    print(f"  {d:<22} {n} arquivos")

## Sobre o tamanho

Os experts são JSON comprimido, uns 90 MB no total — cabem bem no git.

Os adapters é que apertam: cada GGUF tem ~80 MB, e o git guarda **toda versão para
sempre**, então retreinar um adapter e subir de novo soma ao peso em vez de substituir.
Com os 18 experts convertidos, a branch `adapters` passa de 1 GB.

Se isso virar problema, o destino natural pra os pesos é o **Hugging Face Hub**: foi
feito pra modelo, é gratuito pra repositório público e não tem o limite de 100 MB por
arquivo. Os experts podem continuar no git sem incômodo.